# Bronze CDC Ingestion

## Import Helper Functions

In [1]:
from src import *

# while it is allowed to use the import above, it is advised to list out what we have imported
# from src.config_loader import load_config
# from src.spark_sql_magic import sql
# from src.avro_utils import fetch_avro_schema

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
BRONZE_NAMESPACE = cfg["general"]["namespaces"]["bronze"]
CHECKPOINT_BASE = cfg["kafka"]["checkpoint_base"]
KAFKA_BOOTSTRAP = cfg["kafka"]["bootstrap_servers"]
TOPIC_PREFIX = cfg["kafka"]["topic_prefix"]
TOPIC_SUFFIX = set(cfg["kafka"]["topic_suffix"])
APICURIO_URL = cfg["api"]["apicurio_url"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["bronze_cdc_ingestion_backfill"])
        .getOrCreate()
)

## Read Kafka Stream for all topics

In [ ]:
kafka_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
        .option("subscribe", f"{TOPIC_PREFIX}.*")    # <- dynamic subscription
        .option("startingOffsets", "latest")
        .load()
)

## Strip first 5 bytes = Avro wire format
[Link](https://debezium.io/documentation/reference/3.4/configuration/avro.html)

"Apicurio’s Confluent compatibility mode, enabled by setting as-confluent: true, allows Apicurio Registry to serialize Kafka messages using the same wire format as Confluent Schema Registry (including the magic byte and 4-byte schema ID). This enables full interoperability with Confluent clients and tools, making Apicurio a drop-in replacement without requiring changes to producers or consumers."

In [ ]:
kafka_df = (
    kafka_df
    .withColumn("avro_bytes", F.substring(F.col("value"), 6, F.length(F.col("value")) - 5))
    .withColumn("raw_value", F.col("value"))
    .withColumn("spark_ingest_ts", F.current_timestamp())
)

## Function: foreachBatch writer

In [ ]:
def write_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    batch_df = batch_df.withColumn("batch_id", F.lit(str(batch_id)))

    # Process per topic dynamically
    topics = [r.topic for r in batch_df.select("topic").distinct().collect()]

    for topic in topics:
        topic_df = batch_df.filter(F.col("topic") == topic)
        table_name = f"{CATALOG}.{BRONZE_NAMESPACE}.{topic}"
        dlq_table = f"{CATALOG}.{BRONZE_NAMESPACE}_dlq.{topic}_dlq"
        audit_table = f"{CATALOG}.{BRONZE_NAMESPACE}_audit.{topic}_audit"

        schema_dict = fetch_avro_schema(topic, APICURIO_URL, TOPIC_PREFIX)
        schema_json = json.dumps(schema_dict)

        decoded_df = topic_df.select(
            from_avro("avro_bytes", schema_json, {"mode": "PERMISSIVE"}).alias("data"),
            "raw_value",
            "kafka_offset",
            "kafka_partition",
            "kafka_ingest_ts",
            "spark_ingest_ts",
            "batch_id"
        )

        valid_df = decoded_df.filter(F.col("data").isNotNull())
        dlq_df   = decoded_df.filter(F.col("data").isNull())

        valid_df = (
            valid_df
            .select("data.*", "kafka_offset", "kafka_partition", "kafka_ingest_ts", "spark_ingest_ts", "batch_id")
        )

        dlq_df = (
            dlq_df
            .select("raw_value", "kafka_offset", "kafka_partition", "kafka_ingest_ts", "spark_ingest_ts", "batch_id")
        )

        # --- Write bronze and DLQ ---
        if not valid_df.isEmpty():
            valid_df.write.format("iceberg").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
        if not dlq_df.isEmpty():
            dlq_df.write.format("iceberg").mode("append").saveAsTable(dlq_table)

        # --- Audit ---
        audit_partition_df = topic_df.groupBy("kafka_partition").agg(
            F.lit(f"{TOPIC_PREFIX}.{topic}").alias("topic"),
            F.lit(str(batch_id)).alias("batch_id"),
            F.current_timestamp().alias("batch_ts"),
            F.min("kafka_offset").alias("min_offset"),
            F.max("kafka_offset").alias("max_offset"),
            F.count("*").alias("row_count")
        )
        audit_partition_df.write.format("iceberg").mode("append").saveAsTable(audit_table)

        # Print info
        total_rows = audit_partition_df.agg(F.sum("row_count")).collect()[0][0]
        print(f"Batch {batch_id} written for topic {topic} with {total_rows} rows across {audit_partition_df.count()} partitions")
        
        

## Start Streams

In [5]:
query = (
    kafka_df.writeStream
        .foreachBatch(write_batch)
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/{BRONZE_NAMESPACE}/all_topics")
        .trigger(availableNow=True)  # Databricks-friendly
        .start()
)

query.awaitTermination()

Starting Bronze ingestion for customers
Starting Bronze ingestion for sellers
Starting Bronze ingestion for order_items
Starting Bronze ingestion for order_reviews
Starting Bronze ingestion for order_payments
Starting Bronze ingestion for orders
Starting Bronze ingestion for products


[Stage 8:>    (0 + 3) / 3][Stage 10:>   (0 + 1) / 3][Stage 12:>   (0 + 3) / 3]

StreamingQueryException: [STREAM_FAILED] Query [id = 61d6764a-7a56-414c-b750-0d417be282d6, runId = 84ac02c2-ef01-4fef-b4cb-cc3594a623e4] terminated with exception: [FOREACH_BATCH_USER_FUNCTION_ERROR] An error occurred in the user provided function in foreach batch sink. Reason: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/clientserver.py", line 644, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/usr/local/lib/python3.10/dist-packages/pyspark/sql/utils.py", line 165, in call
    raise e
  File "/usr/local/lib/python3.10/dist-packages/pyspark/sql/utils.py", line 162, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_1477/3450889986.py", line 42, in write_batch
    if batch_df.isEmpty():
  File "/usr/local/lib/python3.10/dist-packages/pyspark/sql/classic/dataframe.py", line 282, in isEmpty
    return self.

## Sanity Check Show All Tables

In [ ]:
%%sql
SHOW TABLES IN bronze;

In [ ]:
%%sql
SHOW TABLES IN bronze_audit;

In [ ]:
for item in TOPIC_SUFFIX:
    df = spark.table(f"polaris.bronze.{item}").limit(5)
    df.show()
    df = spark.table(f"polaris.bronze_audit.{item}_audit").limit(5)
    df.show()

In [ ]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  